# 04 — Product-Level Sensitivity

**Project:** SERSA Export Activity and Industrial Consumption Dynamics  
**Author:** Cesar Enrique Banda Martinez  
**Date:** 2026  

---

## Context

Notebook 03 established that at the aggregate level:
- No contemporaneous correlation exists between trade flows and SERSA sales.
- A statistically significant lead-lag relationship emerges at **lag +5 months** across all pairs (r ≈ 0.39–0.41, p < 0.05), with a negative signal at lag +4.
- Both Exportaciones and Importaciones produce virtually identical results — suggesting the two flows are themselves highly correlated and carry the same demand signal.

**Aggregate-level analysis may mask product-level heterogeneity.** Some SKUs may respond strongly to trade activity while others are completely independent. This notebook decomposes the aggregate signal to the individual product level.

### Central question
> Which SKUs show the strongest sensitivity to automotive parts trade flows, and at what lag?

### Approach
1. Build a monthly pivot of SKU-level transaction counts (same method as project 2).
2. Compute cross-correlation between **Exportaciones growth rate** and each SKU's growth rate at lags −6 to +6.
3. Focus on lag +5 (the dominant lag found at aggregate level) as the primary sensitivity metric.
4. Rank SKUs by their correlation at lag +5 and identify the top positive and negative responders.
5. Produce a sensitivity ranking for use in notebook 05.

**Why Exportaciones only?**  
Importaciones produced nearly identical results (r difference < 0.02 across all pairs). Using one flow avoids redundancy. Exportaciones is slightly stronger for transactions and has clearer economic interpretation — it reflects external demand for Mexican automotive output.

---

## 1. Imports

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

---
## 2. Configuration

In [ ]:
RAW_DIR        = os.path.join(os.getcwd(), "..", "data", "raw")
PROCESSED_DIR  = os.path.join(os.getcwd(), "..", "data", "processed")
OUTPUT_FIGURES = os.path.join(os.getcwd(), "..", "outputs", "figures")
OUTPUT_TABLES  = os.path.join(os.getcwd(), "..", "outputs", "tables")

SALES_FILE   = "master_sales.csv"
TARGET_LAG   = 5     # dominant lag from notebook 03
MAX_LAG      = 6
MIN_MONTHS   = 20    # minimum non-zero months for a SKU to be included
LOW_ACTIVITY = 10    # minimum total transactions (same as project 2)

print(f"Target lag : +{TARGET_LAG} months (exports lead sales)")
print(f"Min months : {MIN_MONTHS} months with activity")

---
## 3. Load Data

In [ ]:
# Trade flow data
merged = pd.read_csv(
    os.path.join(PROCESSED_DIR, "merged_monthly_data_enriched.csv"),
    parse_dates=["Month"],
    decimal=","
)

# Raw sales
sales_raw = pd.read_csv(os.path.join(RAW_DIR, SALES_FILE))
sales_raw["Fecha de Consumo"] = pd.to_datetime(sales_raw["Fecha de Consumo"])

print(f"Merged: {merged.shape}  |  {merged['Month'].min().date()} -> {merged['Month'].max().date()}")
print(f"Sales rows: {len(sales_raw):,}")

---
## 4. Build SKU-Level Monthly Pivot

In [ ]:
# Count transactions per SKU per month
monthly_sku = (
    sales_raw
    .groupby([pd.Grouper(key="Fecha de Consumo", freq="MS"), "SKU"])
    .size()
    .reset_index(name="qty")
    .rename(columns={"Fecha de Consumo": "Month"})
)

pivot = (
    monthly_sku
    .pivot(index="Month", columns="SKU", values="qty")
    .fillna(0)
    .astype(int)
)
pivot.index.name = "Month"
pivot.columns.name = None

# Trim to overlap period
overlap_start = merged["Month"].min()
overlap_end   = merged["Month"].max()
pivot = pivot.loc[(pivot.index >= overlap_start) & (pivot.index <= overlap_end)]

print(f"Pivot shape: {pivot.shape}  ->  {pivot.shape[0]} months x {pivot.shape[1]} SKUs")
print(f"Overlap: {pivot.index.min().date()} -> {pivot.index.max().date()}")

---
## 5. Filter SKUs

In [ ]:
# Remove low-activity SKUs
sku_totals   = pivot.sum()
active_skus  = sku_totals[sku_totals >= LOW_ACTIVITY].index

# Remove SKUs with too few active months (sparse series)
active_months = (pivot > 0).sum()
frequent_skus = active_months[active_months >= MIN_MONTHS].index

# Apply both filters
keep_skus = active_skus.intersection(frequent_skus)
pivot_filtered = pivot[keep_skus]

print(f"Total SKUs         : {pivot.shape[1]}")
print(f"After activity filter (>={LOW_ACTIVITY} txns)  : {len(active_skus)}")
print(f"After frequency filter (>={MIN_MONTHS} active months): {len(keep_skus)}")
print(f"Final pivot: {pivot_filtered.shape}")

---
## 6. Compute Growth Rates

In [ ]:
# SKU growth rates
growth_pivot = pivot_filtered.pct_change()
growth_pivot.replace([np.inf, -np.inf], np.nan, inplace=True)

# Export growth rate (from merged)
exports_growth = merged.set_index("Month")["exports_pct"]

# Align index
growth_pivot = growth_pivot.loc[growth_pivot.index.isin(merged["Month"])]
exports_growth = exports_growth.loc[exports_growth.index.isin(growth_pivot.index)]

print(f"Growth pivot: {growth_pivot.shape}")
print(f"Exports growth series: {len(exports_growth)} months")

---
## 7. Cross-Correlation at Target Lag per SKU

For each SKU, we compute Pearson cross-correlation against exports growth rate  
at lags −6 to +6, and extract the correlation at the target lag (+5).

In [ ]:
def cross_corr_at_lag(series_a, series_b, lag):
    """
    Compute Pearson r between series_a[t] and series_b[t+lag].
    Positive lag: series_a leads series_b.
    Returns (r, p_value, n).
    """
    if lag == 0:
        a = series_a.reset_index(drop=True)
        b = series_b.reset_index(drop=True)
    elif lag > 0:
        a = series_a.iloc[:-lag].reset_index(drop=True)
        b = series_b.iloc[lag:].reset_index(drop=True)
    else:
        a = series_a.iloc[-lag:].reset_index(drop=True)
        b = series_b.iloc[:lag].reset_index(drop=True)

    combined = pd.concat([a, b], axis=1).dropna()
    if len(combined) < 5:
        return np.nan, np.nan, 0
    r, p = stats.pearsonr(combined.iloc[:, 0], combined.iloc[:, 1])
    return round(r, 4), round(p, 4), len(combined)

print("cross_corr_at_lag() defined.")

In [ ]:
records = []

for sku in growth_pivot.columns:
    sku_series = growth_pivot[sku]

    # Correlation at target lag (+5)
    r5, p5, n5 = cross_corr_at_lag(exports_growth, sku_series, TARGET_LAG)

    # Contemporaneous correlation (lag 0)
    r0, p0, n0 = cross_corr_at_lag(exports_growth, sku_series, 0)

    # Full cross-correlation to find dominant lag
    best_r, best_lag = 0, 0
    for lag in range(-MAX_LAG, MAX_LAG + 1):
        r, p, n = cross_corr_at_lag(exports_growth, sku_series, lag)
        if not np.isnan(r) and abs(r) > abs(best_r):
            best_r = r
            best_lag = lag

    records.append({
        "SKU"            : sku,
        "r_at_lag5"      : r5,
        "p_at_lag5"      : p5,
        "n_at_lag5"      : n5,
        "sig_at_lag5"    : "Yes" if (p5 is not None and not np.isnan(p5) and p5 < 0.05) else "No",
        "r_contemporaneous": r0,
        "p_contemporaneous": p0,
        "dominant_lag"   : best_lag,
        "max_abs_r"      : round(abs(best_r), 4),
    })

sensitivity = pd.DataFrame(records).sort_values("r_at_lag5", ascending=False).reset_index(drop=True)

print(f"SKUs analyzed: {len(sensitivity)}")
print(f"Significant at lag +5 (p<0.05): {sensitivity['sig_at_lag5'].value_counts().get('Yes', 0)}")
print()
print("Top 15 positive responders at lag +5:")
print(sensitivity.head(15).to_string(index=False))

---
## 8. Sensitivity Ranking Visualization

In [ ]:
# Top 20 positive + top 10 negative
top_pos = sensitivity.head(20)
top_neg = sensitivity.tail(10).sort_values("r_at_lag5")

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Positive
colors_pos = ["#1A5276" if s == "Yes" else "#2C7BB6" for s in top_pos["sig_at_lag5"]]
axes[0].barh(top_pos["SKU"][::-1], top_pos["r_at_lag5"][::-1], color=colors_pos[::-1], alpha=0.85)
axes[0].axvline(x=0, color="black", linewidth=0.8)
axes[0].set_xlabel("Pearson r (Exports lag +5 vs SKU)", fontsize=10)
axes[0].set_title("Top 20 Positive Responders\n(dark = p < 0.05)", fontsize=11, fontweight="bold")
axes[0].grid(axis="x", alpha=0.3)
sns.despine(ax=axes[0])

# Negative
colors_neg = ["#922B21" if s == "Yes" else "#D7191C" for s in top_neg["sig_at_lag5"]]
axes[1].barh(top_neg["SKU"], top_neg["r_at_lag5"], color=colors_neg, alpha=0.85)
axes[1].axvline(x=0, color="black", linewidth=0.8)
axes[1].set_xlabel("Pearson r (Exports lag +5 vs SKU)", fontsize=10)
axes[1].set_title("Top 10 Negative Responders\n(dark = p < 0.05)", fontsize=11, fontweight="bold")
axes[1].grid(axis="x", alpha=0.3)
sns.despine(ax=axes[1])

fig.suptitle(f"SKU Sensitivity to Automotive Exports — Lag +{TARGET_LAG} Months",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_FIGURES, "04_product_sensitivity_ranking.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved: 04_product_sensitivity_ranking.png")

---
## 9. Dominant Lag Distribution Across SKUs

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))

lag_counts = sensitivity["dominant_lag"].value_counts().sort_index()
colors = ["#2C7BB6" if l == 0 else "#D7191C" if l < 0 else "#1A9641" for l in lag_counts.index]
ax.bar(lag_counts.index, lag_counts.values, color=colors, alpha=0.85)

ax.set_xlabel("Dominant Lag (months)  [+ = exports lead SKU]", fontsize=11)
ax.set_ylabel("Number of SKUs", fontsize=11)
ax.set_title("Distribution of Dominant Lag Across All SKUs", fontsize=13, fontweight="bold")
ax.set_xticks(range(-MAX_LAG, MAX_LAG + 1))
ax.grid(axis="y", alpha=0.3)
sns.despine()

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="#D7191C", alpha=0.85, label="Sales leads exports (lag < 0)"),
    Patch(facecolor="#2C7BB6", alpha=0.85, label="Contemporaneous (lag = 0)"),
    Patch(facecolor="#1A9641", alpha=0.85, label="Exports lead sales (lag > 0)"),
]
ax.legend(handles=legend_elements, fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_FIGURES, "04_dominant_lag_distribution_skus.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved: 04_dominant_lag_distribution_skus.png")

---
## 10. Export

In [ ]:
sensitivity.to_csv(
    os.path.join(OUTPUT_TABLES, "04_product_sensitivity_ranking.csv"),
    index=False, decimal=","
)

# Export top SKUs for notebook 05
top_skus_for_index = sensitivity[sensitivity["sig_at_lag5"] == "Yes"].copy()
top_skus_for_index.to_csv(
    os.path.join(OUTPUT_TABLES, "04_significant_skus_lag5.csv"),
    index=False, decimal=","
)

print("Export complete.")
print(f"  04_product_sensitivity_ranking.csv  -> {len(sensitivity)} SKUs")
print(f"  04_significant_skus_lag5.csv        -> {len(top_skus_for_index)} significant SKUs")

---
## 11. Summary

| Output | Description |
|--------|-------------|
| `04_product_sensitivity_ranking.csv` | All SKUs ranked by correlation at lag +5 |
| `04_significant_skus_lag5.csv` | SKUs with p < 0.05 at lag +5 |
| `04_product_sensitivity_ranking.png` | Top positive and negative responders |
| `04_dominant_lag_distribution_skus.png` | Dominant lag distribution across all SKUs |

### Key questions answered here
- Which SKUs react most strongly to changes in automotive export activity 5 months prior?
- How many SKUs show a statistically significant response?
- Is the dominant lag consistent with the aggregate finding (+5 months)?

**Next:** `05_industrial_demand_index.ipynb` — construct a composite Industrial Demand Index from the most trade-sensitive SKUs and compare it against the export series.